This notebook trains and evaluates a random forest model

In [2]:
import numpy as np
import pandas as pd
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    fbeta_score,
    make_scorer,
    confusion_matrix,
)

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

import joblib


In [3]:
#Load train and test
train_df = pd.read_pickle("../data/modeling/train_df.pkl")
test_df  = pd.read_pickle("../data/modeling/test_df.pkl")

print("Train:", train_df.shape)
print("Test :", test_df.shape)


Train: (433620, 98)
Test : (47191, 98)


In [4]:
#Separate Feature Matrix and Target Col
target_col = "default_flag"

X_train = train_df.drop(columns=target_col)
y_train = train_df[target_col]

X_test = test_df.drop(columns=target_col)
y_test = test_df[target_col]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train: (433620, 97)
y_train: (433620,)


In [5]:
# Load preprocessing pipeline (non-XGB)
sk_preprocessor = joblib.load("../data/modeling/sk_preprocessor.pkl")

print("Loaded sklearn preprocessor")

Loaded sklearn preprocessor


In [6]:
#Create TimesSeriesSplit and Evaluation Metric
timesplit = TimeSeriesSplit(n_splits=3)
f2_scorer = make_scorer(fbeta_score, beta=2)

In [7]:
rf_param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [1, 2, 3],
    "model__max_features": [0.1, 0.2, 0.3],
}

In [ ]:

seeds = [42, 84, 126]

out_dir = "../results/rf"


rf_results_summary = []

for seed in seeds:
    print(f"\n================ RF GRID SEARCH (seed={seed}) ================")

    rf_pipe = Pipeline(
        steps=[
            ("preprocess", sk_preprocessor), 
            ("model", RandomForestClassifier(
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=seed,
                n_jobs=4,
            )),
        ]
    )

    rf_grid = GridSearchCV(
        estimator=rf_pipe,
        param_grid=rf_param_grid,
        scoring=f2_scorer,
        cv=timesplit,
        n_jobs=4,
        verbose=0,
        refit=True,
        return_train_score=False,
    )

    rf_grid.fit(X_train, y_train)

    best_params = rf_grid.best_params_
    best_cv_f2 = rf_grid.best_score_
    best_model = rf_grid.best_estimator_

    print(f"[seed={seed}] Best params: {best_params}")
    print(f"[seed={seed}] Best CV F2: {best_cv_f2:.6f}")

    cv_path = os.path.join(out_dir, f"rf_seed{seed}_gridcv.joblib")
    model_path = os.path.join(out_dir, f"rf_seed{seed}_model.joblib")
    joblib.dump(rf_grid, cv_path)
    joblib.dump(best_model, model_path)
    print(f"[seed={seed}] Saved GridSearchCV → {cv_path}")
    print(f"[seed={seed}] Saved best model → {model_path}")

    y_test_pred = best_model.predict(X_test)
    test_f2 = fbeta_score(y_test, y_test_pred, beta=2)
    cm = confusion_matrix(y_test, y_test_pred)

    print(f"[seed={seed}] TEST F2: {test_f2:.6f}")
    print(f"[seed={seed}] Confusion matrix:\n{cm}")

    rf_results_summary.append({
        "seed": seed,
        "cv_f2": float(best_cv_f2),
        "test_f2": float(test_f2),
        "best_params": best_params,
        "gridcv_path": cv_path,
        "model_path": model_path,
    })

summary_df = pd.DataFrame(rf_results_summary)
print("\n================ RF SUMMARY ACROSS SEEDS ================")
display(summary_df[["seed", "cv_f2", "test_f2", "best_params"]])
